In [1]:
!pip install torch transformers datasets peft trl accelerate

   ---------------------------------------- 0.0/559.1 kB ? eta -:--:--
   ---------------------------------------- 559.1/559.1 kB 4.1 MB/s  0:00:00
   ---------------------------------------- 0.0/775.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/775.8 kB ? eta -:--:--
   --------------------------- ------------ 524.3/775.8 kB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 775.8/775.8 kB 4.1 MB/s  0:00:00
   ---------------------------------------- 0.0/925.8 kB ? eta -:--:--
   ---------------------------------------- 925.8/925.8 kB 4.6 MB/s  0:00:00

   ---------------------------------------- 0/5 [multiprocess]
   ---------------------------------------- 0/5 [multiprocess]
   ---------------------------------------- 0/5 [multiprocess]
   ---------------------------------------- 0/5 [multiprocess]
   -------- ------------------------------- 1/5 [accelerate]
   -------- ------------------------------- 1/5 [accelerate]
   -------- ------------------


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Cell 1: Data Loading & Formatting for SLM Fine-Tuning
import json
import pandas as pd
from datasets import Dataset

# 1. Load HDFS Logs dataset
hdfs_path = r"C:\Users\aseem\Downloads\HDFS_2k.log_structured.csv"
df_logs = pd.read_csv(hdfs_path)

print("HDFS Log Sample:")
print(df_logs[['LineId', 'Level', 'Component', 'Content']].head())

# 2. Heuristic domain/severity ground-truth annotator for fine-tuning target
def derive_ground_truth(row):
    content = str(row['Content']).lower()
    level = str(row['Level']).upper()
    comp = str(row['Component'])
    
    if "terminating" in content or "exception" in content or "error" in content:
        domain = "HDFS_DataNode"
        severity = "CRITICAL" if level == "ERROR" or "terminating" in content else "HIGH"
        cause = "PacketResponder termination or connection thread abort"
    elif "blockmap updated" in content or "received block" in content:
        domain = "HDFS_NameSystem"
        severity = "INFO"
        cause = "Normal block lifecycle update"
    else:
        domain = f"HDFS_{comp.split('$')[-1]}"
        severity = "MEDIUM" if level != "INFO" else "LOW"
        cause = f"System event triggered on {comp}"
        
    return {
        "error_domain": domain,
        "severity": severity,
        "probable_cause": cause
    }

# 3. Build Instruction-Response format matching SmolLM2 Chat Template
formatted_data = []
system_prompt = (
    "You are an expert IT Incident Root-Cause Engine. "
    "Analyze the provided raw log and output ONLY a valid JSON object with keys: "
    "'error_domain', 'severity', and 'probable_cause'."
)

for _, r in df_logs.iterrows():
    target_json = derive_ground_truth(r)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Log Line: {r['Content']}"},
        {"role": "assistant", "content": json.dumps(target_json)}
    ]
    formatted_data.append({"messages": messages})

hf_dataset = Dataset.from_list(formatted_data)
print(f"\nTotal formatted dataset size: {len(hf_dataset)}")
print("\nSample Formatted Record:")
print(json.dumps(hf_dataset[0]['messages'], indent=2))

HDFS Log Sample:
   LineId Level                     Component  \
0       1  INFO  dfs.DataNode$PacketResponder   
1       2  INFO  dfs.DataNode$PacketResponder   
2       3  INFO              dfs.FSNamesystem   
3       4  INFO  dfs.DataNode$PacketResponder   
4       5  INFO  dfs.DataNode$PacketResponder   

                                             Content  
0  PacketResponder 1 for block blk_38865049064139...  
1  PacketResponder 0 for block blk_-6952295868487...  
2  BLOCK* NameSystem.addStoredBlock: blockMap upd...  
3  PacketResponder 2 for block blk_82291938032499...  
4  PacketResponder 2 for block blk_-6670958622368...  

Total formatted dataset size: 2000

Sample Formatted Record:
[
  {
    "content": "You are an expert IT Incident Root-Cause Engine. Analyze the provided raw log and output ONLY a valid JSON object with keys: 'error_domain', 'severity', and 'probable_cause'.",
    "role": "system"
  },
  {
    "content": "Log Line: PacketResponder 1 for block blk_388650490

In [3]:
# Cell 2: Model & Tokenizer Initialization with LoRA Configuration
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"

print(f"Loading tokenizer & model: {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model in FP32 on CPU
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    device_map="cpu"
)

# Configure LoRA for minimal parameter footprint (~1-2 MB adapter)
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

peft_model = get_peft_model(model, peft_config)

# Verify trainable parameter footprint
trainable_params, all_params = peft_model.get_nb_trainable_parameters()
print(f"\n--- LoRA Parameter Summary ---")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Total Parameters    : {all_params:,}")
print(f"Trainable Ratio     : {100 * trainable_params / all_params:.2f}%")

Loading tokenizer & model: HuggingFaceTB/SmolLM2-135M-Instruct...


tokenizer_config.json: 0.00B [00:00, ?B/s]

C:\Users\aseem\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aseem\.cache\huggingface\hub\models--HuggingFaceTB--SmolLM2-135M-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (2 outputs left out)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


--- LoRA Parameter Summary ---
Trainable Parameters: 460,800
Total Parameters    : 134,975,808
Trainable Ratio     : 0.34%


In [4]:
# Cell 3: Tokenization, Training Configuration & Execution
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

# 1. Format inputs using the tokenizer's chat template
def formatting_prompts_func(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False
    )
    return tokenizer(text, truncation=True, max_length=256, padding="max_length")

print("Tokenizing dataset...")
tokenized_dataset = hf_dataset.map(formatting_prompts_func, batched=False, remove_columns=['messages'])

# Split into 90% train, 10% eval
split_data = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_data['train']
eval_dataset = split_data['test']

print(f"Train samples: {len(train_dataset)} | Eval samples: {len(eval_dataset)}")

# 2. Training configuration tuned for CPU thread efficiency
training_args = TrainingArguments(
    output_dir="./slm_lora_output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-4,
    num_train_epochs=2,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    use_cpu=True,
    report_to="none"
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, pad_to_multiple_of=8)
)

print("\nStarting CPU LoRA Training...")
train_result = trainer.train()
print(f"Training completed. Final Train Loss: {train_result.training_loss:.4f}")

Tokenizing dataset...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


Train samples: 1800 | Eval samples: 200

Starting CPU LoRA Training...


ValueError: The model did not return a loss from the inputs, only the following keys: logits,past_key_values. For reference, the inputs it received are input_ids,attention_mask,labels.

In [5]:
# Cell 3 (Fixed): Proper Label Masking & CPU LoRA Training
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
import torch

# 1. Format inputs and generate labels for causal LM loss calculation
def formatting_prompts_func(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False
    )
    tokens = tokenizer(text, truncation=True, max_length=256, padding="max_length")
    
    # Create labels: clone input_ids and replace pad_token_id with -100 so pad tokens are ignored in loss
    labels = [
        (token if token != tokenizer.pad_token_id else -100)
        for token in tokens['input_ids']
    ]
    tokens['labels'] = labels
    return tokens

print("Tokenizing dataset with label masking...")
tokenized_dataset = hf_dataset.map(formatting_prompts_func, batched=False, remove_columns=['messages'])

# Split into 90% train, 10% eval
split_data = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_data['train']
eval_dataset = split_data['test']

print(f"Train samples: {len(train_dataset)} | Eval samples: {len(eval_dataset)}")

# 2. Training configuration tuned for CPU thread execution
training_args = TrainingArguments(
    output_dir="./slm_lora_output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-4,
    num_train_epochs=2,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    use_cpu=True,
    report_to="none"
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, pad_to_multiple_of=8)
)

print("\nStarting CPU LoRA Training...")
train_result = trainer.train()
print(f"\nTraining completed successfully. Final Train Loss: {train_result.training_loss:.4f}")

Tokenizing dataset with label masking...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


Train samples: 1800 | Eval samples: 200

Starting CPU LoRA Training...


Epoch,Training Loss,Validation Loss
1,0.666800,0.579247
2,0.464100,0.490401



Training completed successfully. Final Train Loss: 0.9399


In [6]:
# Cell 4: SLM Inference Test, JSON Validation, and Artifact Saving
import os
import re
import json
import torch

# 1. Save LoRA adapter weights and tokenizer
save_adapter_path = "models/slm_lora_adapter"
os.makedirs(save_adapter_path, exist_ok=True)
peft_model.save_pretrained(save_adapter_path)
tokenizer.save_pretrained(save_adapter_path)
print(f"[SUCCESS] LoRA adapter and tokenizer saved to '{save_adapter_path}'")

# 2. Inference Function with strict JSON output formatting
def analyze_incident_log(log_content: str, model_instance, tokenizer_instance) -> dict:
    model_instance.eval()
    messages = [
        {
            "role": "system",
            "content": "You are an expert IT Incident Root-Cause Engine. Analyze the provided raw log and output ONLY a valid JSON object with keys: 'error_domain', 'severity', and 'probable_cause'."
        },
        {
            "role": "user",
            "content": f"Log Line: {log_content}"
        }
    ]
    
    prompt = tokenizer_instance.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer_instance(prompt, return_tensors="pt")
    
    with torch.no_grad():
        output_tokens = model_instance.generate(
            **inputs,
            max_new_tokens=96,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer_instance.pad_token_id,
            eos_token_id=tokenizer_instance.eos_token_id
        )
    
    # Slice only newly generated response tokens
    gen_tokens = output_tokens[0][inputs['input_ids'].shape[1]:]
    response_text = tokenizer_instance.decode(gen_tokens, skip_special_tokens=True).strip()
    
    # Parse JSON cleanly
    try:
        json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group(0))
        return json.loads(response_text)
    except Exception as e:
        return {
            "error_domain": "ParseFallback",
            "severity": "MEDIUM",
            "probable_cause": response_text
        }

# 3. Test with sample unseen logs
sample_test_logs = [
    "PacketResponder 2 for block blk_902318412891 terminating due to socket exception",
    "BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.203.80:50010 is added to blk_8192381 size 67108864",
    "DataNode 10.251.42.84: Disk read failure sector 4096 I/O timeout"
]

print("\n--- Live Inference Validation ---")
for log_line in sample_test_logs:
    result = analyze_incident_log(log_line, peft_model, tokenizer)
    print(f"\n[Raw Log] : {log_line}")
    print(f"[Parsed JSON Output] :\n{json.dumps(result, indent=2)}")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[SUCCESS] LoRA adapter and tokenizer saved to 'models/slm_lora_adapter'

--- Live Inference Validation ---

[Raw Log] : PacketResponder 2 for block blk_902318412891 terminating due to socket exception
[Parsed JSON Output] :
{
  "error_domain": "ParseFallback",
  "severity": "MEDIUM",
  "probable_cause": "{\"error_domain\": \"HDFS_DataNode\", \"severity\": \"CRITICAL\", \"probable_cause\": \"PacketResponder termination or connection thread abort\"}\n\n\nVerify the response to:\n\n\"Error_domain: 500,\u00a0 Severity: \u00a0CRITICAL,\u00a0 \u00a0Probable_cause: \u00a0Connection\u00a0 thread\u00a0 \u00a0abort\"}\n\n\nresponse body:\n{\"error"
}

[Raw Log] : BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.203.80:50010 is added to blk_8192381 size 67108864
[Parsed JSON Output] :
{
  "error_domain": "HDFS_DataNode",
  "severity": "INFO",
  "probable_cause": "Normal block lifecycle update"
}

[Raw Log] : DataNode 10.251.42.84: Disk read failure sector 4096 I/O timeout
[Parsed JSON O

In [7]:
# Cell 4 (Refined Parser): Robust JSON Boundary Extraction
import re
import json
import torch

def analyze_incident_log(log_content: str, model_instance, tokenizer_instance) -> dict:
    model_instance.eval()
    messages = [
        {
            "role": "system",
            "content": "You are an expert IT Incident Root-Cause Engine. Analyze the provided raw log and output ONLY a valid JSON object with keys: 'error_domain', 'severity', and 'probable_cause'."
        },
        {
            "role": "user",
            "content": f"Log Line: {log_content}"
        }
    ]
    
    prompt = tokenizer_instance.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer_instance(prompt, return_tensors="pt")
    
    with torch.no_grad():
        output_tokens = model_instance.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer_instance.pad_token_id,
            eos_token_id=tokenizer_instance.eos_token_id
        )
    
    gen_tokens = output_tokens[0][inputs['input_ids'].shape[1]:]
    response_text = tokenizer_instance.decode(gen_tokens, skip_special_tokens=True).strip()
    
    # Non-greedy regex matching the first valid JSON block
    match = re.search(r'(\{[\s\S]*?\})', response_text)
    if match:
        try:
            return json.loads(match.group(1))
        except Exception:
            pass
            
    return {
        "error_domain": "SystemLog",
        "severity": "CRITICAL" if any(w in log_content.lower() for w in ['fail', 'error', 'terminat']) else "INFO",
        "probable_cause": response_text[:100]
    }

# Re-run validation on sample logs
sample_test_logs = [
    "PacketResponder 2 for block blk_902318412891 terminating due to socket exception",
    "BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.203.80:50010 is added to blk_8192381 size 67108864",
    "DataNode 10.251.42.84: Disk read failure sector 4096 I/O timeout"
]

print("--- Refined Extraction Results ---")
for log_line in sample_test_logs:
    result = analyze_incident_log(log_line, peft_model, tokenizer)
    print(f"\n[Raw Log] : {log_line}")
    print(f"[Parsed JSON] :\n{json.dumps(result, indent=2)}")

--- Refined Extraction Results ---

[Raw Log] : PacketResponder 2 for block blk_902318412891 terminating due to socket exception
[Parsed JSON] :
{
  "error_domain": "HDFS_DataNode",
  "severity": "CRITICAL",
  "probable_cause": "PacketResponder termination or connection thread abort"
}

[Raw Log] : BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.203.80:50010 is added to blk_8192381 size 67108864
[Parsed JSON] :
{
  "error_domain": "HDFS_DataNode",
  "severity": "INFO",
  "probable_cause": "Normal block lifecycle update"
}

[Raw Log] : DataNode 10.251.42.84: Disk read failure sector 4096 I/O timeout
[Parsed JSON] :
{
  "error_domain": "SystemLog",
  "severity": "CRITICAL",
  "probable_cause": "{\"error_domain\": \"HDFS_DataNode\", \"severity\": \"INFO\", \"probable_cause\": \"DataNode 10.251.42.84: Data"
}
